# 1.DataSet Processing

In [1]:
import json
import os
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from ast import literal_eval

In [2]:
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	tokens = word_tokenize(text.lower())
	filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	return ' '.join(filtered_tokens)
	
# Load JSON data
def load_data(filepath):
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data

[nltk_data] Downloading package punkt to /Users/chenluyao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
train_claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')
dev_claims_data = load_data('data/dev-claims.json')
evidence_map = load_data('data/curated/preprocessed_evidence_map.json')

In [4]:
evidence_df = pd.DataFrame(evidence_map.items(), columns=['id', 'evidence'])
evidence_df

,id,evidence
0,evidence-0,john bennet law english entrepreneur agricultu...
1,evidence-1,lindberg began profession career age 16 eventu...
2,evidence-2,boston ladi cambridg vampir weekend
3,evidence-3,gerald franci goyer born octob 20 1936 profess...
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...
...,...,...
1208822,evidence-1208822,also properti contribut garag apart
1208823,evidence-1208823,class fn org fyrd 6110 volda
1208824,evidence-1208824,dragon storm game game collect card game
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...


In [5]:
data_for_dataframe = []
for claim_id, claim_details in train_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids
        })

# Create DataFrame
train_claims_df = pd.DataFrame(data_for_dataframe)
train_claims_df 

,claim,evidence
0,scientif evid co2 pollut higher co2 concentr a...,"[evidence-442946, evidence-1194317, evidence-1..."
1,el niño drove record high global temperatur su...,"[evidence-338219, evidence-1127398]"
2,1946 pdo switch cool phase,"[evidence-530063, evidence-984887]"
3,weather channel john coleman provid evid convi...,"[evidence-1177431, evidence-782448, evidence-5..."
4,januari 2008 cap 12 month period global temper...,"[evidence-1010750, evidence-91661, evidence-72..."
...,...,...
1223,climat scientist say aspect case hurrican harv...,"[evidence-1055682, evidence-1047356, evidence-..."
1224,5th assess report 2013 ipcc estim human emiss ...,[evidence-916755]
1225,sinc mid 1970 global temperatur warm around de...,"[evidence-403673, evidence-889933, evidence-11..."
1226,abnorm temperatur spike februari earlier month...,"[evidence-97375, evidence-562427, evidence-521..."


In [6]:
all_texts = train_claims_df['claim'].tolist() + evidence_df['evidence'].tolist()

In [ ]:
# tagged_data = [TaggedDocument(words=_d.split(), tags=[str(i)]) for i, _d in enumerate(all_texts)]

# model = Doc2Vec(vector_size=50, min_count=1, epochs=20)
  
# model.build_vocab(tagged_data)
# model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)
# model.save("d2v.model")

In [9]:
# import random

# # Vectorization
# vectorizer = TfidfVectorizer()
# sample_texts = random.sample(all_texts, 500)

# # Fit the vectorizer on both claims and evidences
# vectorizer.fit(sample_texts) 
# claim_vec = vectorizer.transform(claims_df['claim']).toarray() 
# claims_df['claim_tfidf'] = list(claim_vec) 
# claim_vec.shape

# evidence_vec = vectorizer.transform(evidence_df['evidence']).toarray()
# evidence_vec.shape

In [7]:
model= Doc2Vec.load("d2v.model")

In [8]:
# train_claims_df["vector"] = ""
# for i in range(train_claims_df.shape[0]):
#     inferred_vector = model.infer_vector(train_claims_df["claim"][i].split())
#     train_claims_df["vector"][i] = inferred_vector
# train_claims_df.to_csv('data/curated/train_claim_vector.csv', index=False)
# train_claims_df 

In [10]:
# evidence_df["vector"] = ""
# for i in range(evidence_df.shape[0]):
#     inferred_vector = model.infer_vector(evidence_df["evidence"][i].split())
#     evidence_df["vector"][i] = inferred_vector
# evidence_df.to_csv('data/curated/evidence_vector.csv', index=False)

evidence_df = pd.read_csv('data/curated/evidence_vector.csv', converters={
    'vector': lambda x: np.array(x.strip("[]").split(), dtype='float')})
evidence_df

,id,evidence,vector
0,evidence-0,john bennet law english entrepreneur agricultu...,"[-0.08329561, -0.15579118, -0.17553866, -0.129..."
1,evidence-1,lindberg began profession career age 16 eventu...,"[0.19000088, 0.15242222, -0.45844978, -0.01609..."
2,evidence-2,boston ladi cambridg vampir weekend,"[0.05613353, -0.1325368, 0.15981574, 0.0887365..."
3,evidence-3,gerald franci goyer born octob 20 1936 profess...,"[0.12277374, -0.09053773, -0.3347684, -0.28721..."
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...,"[0.21676677, -0.151133, -0.48791853, 0.1081487..."
...,...,...,...
1208822,evidence-1208822,also properti contribut garag apart,"[0.08558074, -0.04533045, -0.14334618, -0.0497..."
1208823,evidence-1208823,class fn org fyrd 6110 volda,"[0.07639017, -0.07667744, -0.3351644, -0.06447..."
1208824,evidence-1208824,dragon storm game game collect card game,"[0.14750834, -0.05903809, -0.40002766, -0.1349..."
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...,"[0.4817398, -0.25093532, -0.49334916, -0.16006..."


In [14]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'evidence': eids
        })
    
# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df 

,claim_id,claim,evidence
0,claim-752,south australia expens electr world,"[evidence-67732, evidence-572512]"
1,claim-375,3 per cent total annual global emiss carbon di...,"[evidence-996421, evidence-1080858, evidence-2..."
2,claim-1266,mean world 1c warmer time,"[evidence-889933, evidence-694262]"
3,claim-871,happen zika may also good model second worri e...,"[evidence-422399, evidence-702226, evidence-28..."
4,claim-2164,greenland lost tini fraction ice mass,"[evidence-52981, evidence-264761, evidence-947..."
...,...,...,...
149,claim-2400,suddenli label co2 pollut disservic ga play en...,"[evidence-409365, evidence-127519, evidence-85..."
150,claim-204,natur orbit driven warm atmospher carbon dioxi...,"[evidence-368192, evidence-261690, evidence-20..."
151,claim-1426,mani world coral reef alreadi barren state con...,"[evidence-1124018, evidence-995813, evidence-1..."
152,claim-698,recent studi led lawrenc livermor nation labor...,[evidence-660755]


In [15]:
dev_claims_df["vector"] = ""
for i in range(dev_claims_df.shape[0]):
    inferred_vector = model.infer_vector(dev_claims_df["claim"][i].split())
    dev_claims_df["vector"][i] = inferred_vector
dev_claims_df.to_csv('data/curated/dev_claim_vector.csv', index=False)
dev_claims_df 

,claim_id,claim,evidence,vector
0,claim-752,south australia expens electr world,"[evidence-67732, evidence-572512]","[-0.14783166, -0.13225463, 0.076199934, -0.157..."
1,claim-375,3 per cent total annual global emiss carbon di...,"[evidence-996421, evidence-1080858, evidence-2...","[0.074097365, -0.21100527, -0.3013799, -0.1396..."
2,claim-1266,mean world 1c warmer time,"[evidence-889933, evidence-694262]","[0.0033973104, -0.18324292, -0.15176894, -0.08..."
3,claim-871,happen zika may also good model second worri e...,"[evidence-422399, evidence-702226, evidence-28...","[0.033817377, 0.3418055, -0.12446742, 0.247180..."
4,claim-2164,greenland lost tini fraction ice mass,"[evidence-52981, evidence-264761, evidence-947...","[0.11179431, -0.0334027, -0.15025018, -0.08969..."
...,...,...,...,...
149,claim-2400,suddenli label co2 pollut disservic ga play en...,"[evidence-409365, evidence-127519, evidence-85...","[0.23459642, -0.050042894, -0.007040568, -0.24..."
150,claim-204,natur orbit driven warm atmospher carbon dioxi...,"[evidence-368192, evidence-261690, evidence-20...","[0.071366325, -0.11404269, -0.10364698, -0.042..."
151,claim-1426,mani world coral reef alreadi barren state con...,"[evidence-1124018, evidence-995813, evidence-1...","[-0.11993008, 0.1760608, -0.07409863, -0.12002..."
152,claim-698,recent studi led lawrenc livermor nation labor...,[evidence-660755],"[0.2982595, 0.4369322, -0.18004608, -0.0627699..."


In [13]:
X = np.array(dev_claims_df['vector'].values.tolist())
y = np.array(evidence_df['vector'].values.tolist())
sim = cosine_similarity(X, y)
print(sim.shape)
    

[[ 0.14903344 -0.06424405  0.05079841 ...  0.00578057  0.01542425
   0.00556075]
 [-0.098553    0.83373462 -0.41138423 ...  0.84804849  0.65151597
   0.03489912]
 [ 0.05295772  0.26753896 -0.16369237 ...  0.34944648  0.39603415
   0.08428407]
 ...
 [-0.14019277  0.08372977  0.12076655 ... -0.09482686  0.00892078
   0.12079354]
 [ 0.05272074  0.1886617  -0.28210135 ...  0.18556082 -0.00522221
  -0.05158551]
 [-0.01303664  0.33250677 -0.18119427 ...  0.39189777  0.26040172
   0.22780153]]
(154, 1208827)


In [26]:
# get top 5 evidence with highest similarity score with the claim
data = np.zeros((sim.shape[0], 5))
for i in range(sim.shape[0]):
	data[i] = np.argpartition(sim[i], -5)[-5:]
data = data.astype(np.int32)

dev_claims_df['top5_evidence_id'] = data.tolist()
dev_claims_df

,claim_id,claim,evidence,vector,top5_evidence_id
0,claim-752,south australia expens electr world,"[evidence-67732, evidence-572512]","[-0.14783166, -0.13225463, 0.076199934, -0.157...","[882428, 649294, 641060, 1182886, 805502]"
1,claim-375,3 per cent total annual global emiss carbon di...,"[evidence-996421, evidence-1080858, evidence-2...","[0.074097365, -0.21100527, -0.3013799, -0.1396...","[177788, 926098, 232504, 581681, 8029]"
2,claim-1266,mean world 1c warmer time,"[evidence-889933, evidence-694262]","[0.0033973104, -0.18324292, -0.15176894, -0.08...","[422585, 154455, 177516, 578206, 490494]"
3,claim-871,happen zika may also good model second worri e...,"[evidence-422399, evidence-702226, evidence-28...","[0.033817377, 0.3418055, -0.12446742, 0.247180...","[915283, 768114, 1044283, 596925, 798037]"
4,claim-2164,greenland lost tini fraction ice mass,"[evidence-52981, evidence-264761, evidence-947...","[0.11179431, -0.0334027, -0.15025018, -0.08969...","[30741, 1066768, 952358, 743281, 691825]"
...,...,...,...,...,...
149,claim-2400,suddenli label co2 pollut disservic ga play en...,"[evidence-409365, evidence-127519, evidence-85...","[0.23459642, -0.050042894, -0.007040568, -0.24...","[553483, 1149974, 857419, 185223, 764353]"
150,claim-204,natur orbit driven warm atmospher carbon dioxi...,"[evidence-368192, evidence-261690, evidence-20...","[0.071366325, -0.11404269, -0.10364698, -0.042...","[670135, 1199262, 331280, 11958, 1109021]"
151,claim-1426,mani world coral reef alreadi barren state con...,"[evidence-1124018, evidence-995813, evidence-1...","[-0.11993008, 0.1760608, -0.07409863, -0.12002...","[798917, 887298, 244858, 577021, 1057505]"
152,claim-698,recent studi led lawrenc livermor nation labor...,[evidence-660755],"[0.2982595, 0.4369322, -0.18004608, -0.0627699...","[751260, 1104028, 778957, 1091740, 394932]"


In [27]:
dev_claims_df.to_csv("data/curated/dev_evidence_retrieval.csv", index=False)

### Claim Classification

In [29]:
dev_claims_df = pd.read_csv('data/curated/dev_evidence_retrieval.csv', converters={
    'top5_evidence_id': lambda x: np.array(x.strip("[]").split(','), dtype='int')})
dev_claims_df

,claim_id,claim,evidence,vector,top5_evidence_id
0,claim-752,south australia expens electr world,"['evidence-67732', 'evidence-572512']",[-0.14783166 -0.13225463 0.07619993 -0.157366...,"[882428, 649294, 641060, 1182886, 805502]"
1,claim-375,3 per cent total annual global emiss carbon di...,"['evidence-996421', 'evidence-1080858', 'evide...",[ 0.07409737 -0.21100527 -0.3013799 -0.139605...,"[177788, 926098, 232504, 581681, 8029]"
2,claim-1266,mean world 1c warmer time,"['evidence-889933', 'evidence-694262']",[ 0.00339731 -0.18324292 -0.15176894 -0.085985...,"[422585, 154455, 177516, 578206, 490494]"
3,claim-871,happen zika may also good model second worri e...,"['evidence-422399', 'evidence-702226', 'eviden...",[ 0.03381738 0.3418055 -0.12446742 0.247180...,"[915283, 768114, 1044283, 596925, 798037]"
4,claim-2164,greenland lost tini fraction ice mass,"['evidence-52981', 'evidence-264761', 'evidenc...",[ 0.11179431 -0.0334027 -0.15025018 -0.089694...,"[30741, 1066768, 952358, 743281, 691825]"
...,...,...,...,...,...
149,claim-2400,suddenli label co2 pollut disservic ga play en...,"['evidence-409365', 'evidence-127519', 'eviden...",[ 0.23459642 -0.05004289 -0.00704057 -0.245093...,"[553483, 1149974, 857419, 185223, 764353]"
150,claim-204,natur orbit driven warm atmospher carbon dioxi...,"['evidence-368192', 'evidence-261690', 'eviden...",[ 0.07136633 -0.11404269 -0.10364698 -0.042170...,"[670135, 1199262, 331280, 11958, 1109021]"
151,claim-1426,mani world coral reef alreadi barren state con...,"['evidence-1124018', 'evidence-995813', 'evide...",[-0.11993008 0.1760608 -0.07409863 -0.120029...,"[798917, 887298, 244858, 577021, 1057505]"
152,claim-698,recent studi led lawrenc livermor nation labor...,['evidence-660755'],[ 0.2982595 0.4369322 -0.18004608 -0.062769...,"[751260, 1104028, 778957, 1091740, 394932]"


In [30]:
df = dev_claims_df[["claim_id", "top5_evidence_id"]]
df

,claim_id,top5_evidence_id
0,claim-752,"[882428, 649294, 641060, 1182886, 805502]"
1,claim-375,"[177788, 926098, 232504, 581681, 8029]"
2,claim-1266,"[422585, 154455, 177516, 578206, 490494]"
3,claim-871,"[915283, 768114, 1044283, 596925, 798037]"
4,claim-2164,"[30741, 1066768, 952358, 743281, 691825]"
...,...,...
149,claim-2400,"[553483, 1149974, 857419, 185223, 764353]"
150,claim-204,"[670135, 1199262, 331280, 11958, 1109021]"
151,claim-1426,"[798917, 887298, 244858, 577021, 1057505]"
152,claim-698,"[751260, 1104028, 778957, 1091740, 394932]"
